# Imports

In [1]:
import numpy as np
from matplotlib import pyplot as plt

# Constants

In [2]:
P_ARRAY = np.arange(0, 0.105, 0.005)
N_ARRAY = np.arange(2, 16, 1)
ITERS = 100
TRIALS = 50
print(P_ARRAY)
print(N_ARRAY)

[0.    0.005 0.01  0.015 0.02  0.025 0.03  0.035 0.04  0.045 0.05  0.055
 0.06  0.065 0.07  0.075 0.08  0.085 0.09  0.095 0.1  ]
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15]


# Functions

In [3]:
def getAllBitstrings(n:int, includeZero:bool=True) -> list:
    '''
    Returns a list of all bitstrings of length n.
    Args:
        n (int): the length of the bitstrings.
        includeZero (bool): whether to include all-zeros string (default: True).
    Returns:
        A list of all bitstrings of length n.
    '''

    bitstrings = []
    start = 0 if includeZero else 1
    for x in range(start, 2**n):
        bitstrings.append(np.array(list(map(int, list(f'{x:0{n}b}')))))
    return bitstrings

getAllBitstrings(3)

[array([0, 0, 0]),
 array([0, 0, 1]),
 array([0, 1, 0]),
 array([0, 1, 1]),
 array([1, 0, 0]),
 array([1, 0, 1]),
 array([1, 1, 0]),
 array([1, 1, 1])]

In [4]:
def getOrthogonalSet(secretString:np.array) -> list:
    '''
    Returns a list of all bitstrings that are orthogonal to the secret string.
    Args:
        secretString (np.array): the secret string.
    Returns:
        A list of all bitstrings that are orthogonal to the given secret string.
    '''

    orthogonalSet = []
    n = len(secretString)
    bitstrings = getAllBitstrings(n)
    for bits in bitstrings:
        if sum(bits * secretString) % 2 == 0:
            orthogonalSet.append(bits)
    return orthogonalSet

print(getOrthogonalSet([1, 1, 1]))
print(getOrthogonalSet([1, 0, 1]))

[array([0, 0, 0]), array([0, 1, 1]), array([1, 0, 1]), array([1, 1, 0])]
[array([0, 0, 0]), array([0, 1, 0]), array([1, 0, 1]), array([1, 1, 1])]


In [5]:
def solveSubsystem(subsystem:np.array) -> np.array:
    '''
    Solves the given subsystem of equations.
    Args:
        subsystem (np.array): the subsystem of equations to solve.
    Returns:
        A bitstring that satisfies the subsystem of equations,
        or the all-zeros string if no such bitstring exists.
    '''

    n = len(subsystem[0])
    zeros = np.zeros(n-1, int)
    bitstrings = getAllBitstrings(n, False)
    for bits in bitstrings:
        if np.allclose((subsystem @ bits) % 2, zeros):
            return bits
    return np.zeros(n, int)

print(solveSubsystem(np.array([np.array([1, 0, 1]), np.array([1, 1, 0])])))
print(solveSubsystem(np.array([np.array([0, 1, 1]), np.array([0, 1, 0])])))

[1 1 1]
[1 0 0]


In [6]:
def validateSubsystem(subsystem:np.array, secretString:np.array) -> np.array:
    '''
    Validate that the provided secret string solves the given subsystem.
    Args:
        subsystem (np.array): the subsystem of equations to solve.
        secretString (np.array): the secret string.
    Returns:
        The secret string if it satisfies the subsystem of equations,
        or the all-zeros string if no such bitstring exists.
    '''

    n = len(subsystem[0])
    zeros = np.zeros(n-1, int)
    if np.allclose((subsystem @ secretString) % 2, zeros):
        return secretString
    return np.zeros(n, int)

print(solveSubsystem(np.array([np.array([1, 0, 1]), np.array([1, 1, 0])])))
print(solveSubsystem(np.array([np.array([0, 1, 1]), np.array([0, 1, 0])])))

[1 1 1]
[1 0 0]


In [ ]:
def solveNoisySystem(
        noisySamples:np.array,
        secretString:np.array,
        trials:int=TRIALS,
        solveSystem:bool=True
    ) -> bool:
    '''
    Solves the given noisy system of equations.
    Args:
        noisySamples (np.array): the noisy samples of equations.
        secretString (np.array): the secret string.
        trials (int): the number of random subsystems to try (default: 50).
        solveSystem (bool): Whether to solve or validate the system.
            Validating uses knowledge of s for computational efficiency.
    Returns:
        True if the secret string can be recovered, False otherwise.
    '''

    n = len(secretString)
    for i in range(trials):
        indices = np.random.choice(len(noisySamples), n-1, replace=False)
        subsystem = np.array([noisySamples[i] for i in indices])
        for element in subsystem:
            for j, value in enumerate(element):
                if value == -1:
                    element[j] = np.random.choice([0, 1])
        if solveSystem:
            solution = solveSubsystem(subsystem)
        else:
            solution = validateSubsystem(subsystem, secretString)
        if np.allclose(solution, secretString):
            return True
    return False

solveNoisySystem(
    [
        np.array([1, 0, 1]),
        np.array([0, 0, -1]),
        np.array([1, 0, -1]),
        np.array([-1, 0, 1]),
        np.array([1, 0, 1]),
        np.array([1, 0, 1]),
        np.array([0, 1, -1]),
        np.array([-1, -1, 0]),
        np.array([1, 1, 0]),
        np.array([1, 0, 1]),
        np.array([1, 1, -1]),
        np.array([1, 0, 1]),
        np.array([1, -1, 0]),
        np.array([0, -1, 0]),
        np.array([1, 1, 0]),
        np.array([1, -1, 1]),
        np.array([0, -1, 0]),
        np.array([0, 0, 0]),
        np.array([1, -1, -1]),
        np.array([0, 0, 0]),
        np.array([0, 0, -1]),
        np.array([0, 1, -1]),
        np.array([1, 1, 0]),
    ], [1, 1, 1]
)

True

# Experiment

In [8]:
try:
    successHeatmap = np.load('data/erasureSuccessHeatmap.npy')
    with open('data/erasureIterations.txt', 'r') as file:
        start = int(file.read())
except:
    successHeatmap = np.zeros((len(N_ARRAY), len(P_ARRAY)))
    start = 0
for iter in range(start, ITERS):
    print(f'Iteration {iter}')
    for i, n in enumerate(N_ARRAY):
        print(f'n = {n}:\t', end=' ')
        for j, p in enumerate(P_ARRAY):
            if p in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
                print(f'p = {np.round(p, 2)}', end=' ')
            shots = n + 20  # n + 20 chosen to make P(failure|p=0) < 1/1,000,000
            secretString = np.array([1] * n)
            orthogonalSet = getOrthogonalSet(secretString)
            noisySamples = []
            for shot in range(shots):
                sample = orthogonalSet[
                    np.random.choice(range(len(orthogonalSet)))
                ]
                eraseProbs = np.random.sample(n)
                eraseArray = np.array(list(map(bool, eraseProbs < p)))
                noisySample = sample.copy()
                for k, erase in enumerate(eraseArray):
                    if erase:
                        noisySample[k] = -1
                noisySamples.append(noisySample)
            successHeatmap[i,j] += solveNoisySystem(
                np.array(noisySamples), secretString, solveSystem=False
            )
        print('\n', end='')
    np.save('data/erasureSuccessHeatmap.npy', successHeatmap)
    with open('data/erasureIterations.txt', 'w+') as file:
        file.write(str(iter + 1))

successHeatmap

Iteration 0
n = 2:	 p = 0.0 p = 0.1 
n = 3:	 p = 0.0 p = 0.1 
n = 4:	 p = 0.0 p = 0.1 
n = 5:	 p = 0.0 p = 0.1 
n = 6:	 p = 0.0 p = 0.1 
n = 7:	 p = 0.0 p = 0.1 
n = 8:	 p = 0.0 p = 0.1 
n = 9:	 p = 0.0 p = 0.1 
n = 10:	 p = 0.0 p = 0.1 
n = 11:	 p = 0.0 p = 0.1 
n = 12:	 p = 0.0 p = 0.1 
n = 13:	 p = 0.0 p = 0.1 
n = 14:	 p = 0.0 p = 0.1 
n = 15:	 p = 0.0 p = 0.1 
Iteration 1
n = 2:	 p = 0.0 p = 0.1 
n = 3:	 p = 0.0 p = 0.1 
n = 4:	 p = 0.0 p = 0.1 
n = 5:	 p = 0.0 p = 0.1 
n = 6:	 p = 0.0 p = 0.1 
n = 7:	 p = 0.0 p = 0.1 
n = 8:	 p = 0.0 p = 0.1 
n = 9:	 p = 0.0 p = 0.1 
n = 10:	 p = 0.0 p = 0.1 
n = 11:	 p = 0.0 p = 0.1 
n = 12:	 p = 0.0 p = 0.1 
n = 13:	 p = 0.0 p = 0.1 
n = 14:	 p = 0.0 p = 0.1 
n = 15:	 p = 0.0 p = 0.1 
Iteration 2
n = 2:	 p = 0.0 p = 0.1 
n = 3:	 p = 0.0 p = 0.1 
n = 4:	 p = 0.0 p = 0.1 
n = 5:	 p = 0.0 p = 0.1 
n = 6:	 p = 0.0 p = 0.1 
n = 7:	 p = 0.0 p = 0.1 
n = 8:	 p = 0.0 p = 0.1 
n = 9:	 p = 0.0 p = 0.1 
n = 10:	 p = 0.0 p = 0.1 
n = 11:	 p = 0.0 p = 0.1 


array([[100., 100., 100., 100., 100., 100., 100., 100., 100., 100., 100.,
        100., 100., 100., 100., 100., 100., 100., 100., 100., 100.],
       [100., 100., 100., 100., 100., 100., 100., 100., 100., 100., 100.,
        100., 100., 100., 100., 100., 100., 100., 100., 100., 100.],
       [100., 100., 100., 100., 100., 100., 100., 100., 100., 100., 100.,
        100., 100., 100., 100., 100., 100., 100., 100., 100., 100.],
       [100., 100., 100., 100., 100., 100., 100., 100., 100., 100., 100.,
        100., 100., 100., 100., 100., 100., 100., 100., 100., 100.],
       [100., 100., 100., 100., 100., 100., 100., 100., 100., 100., 100.,
        100., 100., 100., 100., 100., 100., 100., 100., 100., 100.],
       [100., 100., 100., 100., 100., 100., 100., 100., 100., 100., 100.,
        100., 100., 100., 100., 100., 100., 100., 100., 100.,  99.],
       [100., 100., 100., 100., 100., 100., 100., 100., 100., 100., 100.,
        100., 100., 100., 100., 100., 100.,  98.,  99.,  99.,  93.],